# SSTCa2 — pipeline driver

Notebook entry point for the refactored SSTCa2 analysis. Each cell runs one stage; you can stop at any point and inspect state in the live kernel.

**Architecture**
- `SSTCa2_config.PipelineConfig` — all user-tunable switches (defaults match legacy `SSTCa2_main.py`).
- `SSTCa2_loader.load_all_mice(cfg)` — returns a `SimpleNamespace` with every metadata dict, session dict, mapping accumulator, and engram-pass result the legacy script produced.
- `SSTCa2_pipeline.*` — thin wrappers around the analysis modules (`SSTCa2_population`, `SSTCa2_isomap`, `SSTCa2_epoch_analysis`, ...).

**Globals injection.** After `load_all_mice`, the cell
```python
globals().update(vars(ds))
```
brings every per-mouse dict (`TFC_cond`, `Test_B`, `engram_id`, every `mappings_all_*`, every `dpath_*`, ...) into the notebook's top-level namespace. Both `ds.TFC_cond` and bare `TFC_cond` then reference the same object, so legacy debug snippets keep working unchanged.

## 1. Setup

`%autoreload 2` rebuilds modules on edit. Heavy code lives in `SSTCa2_*.py` and is reloaded automatically; the dataset itself is **not** rebuilt — you keep your loaded `ds`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np
import matplotlib.pyplot as plt

import SSTCa2_config
import SSTCa2_loader
import SSTCa2_pipeline
from SSTCa2_config import PipelineConfig
from SSTCa2_loader import load_all_mice
from SSTCa2_pipeline import (
    resolve_continuity_params,
    engram_idx_by_mouse,
    run_engram_sanity_plots,
    run_population_pca,
    run_population_pca_all_modes,
    run_isomap,
    run_epoch_pv,
    run_cross_session_epoch_pv,
)

## 2. Build config

Edit fields here to override defaults. All ~90 switches from legacy `SSTCa2_main.py` are exposed.

In [ ]:
cfg = PipelineConfig(
    # DEBUG=True,                # restrict to one mouse for quick smoke tests
    # plot_pf_raw_maps=True,     # heavy per-cell PF plots
    # optimize_parameters=True,  # run raw-S decoder Optuna study
    # enable_population_curve=True,
)
print(cfg)

## 3. Load all mice

Builds CrossReg + Session objects, runs per-mapping accumulators, the unified engram-identity pass, and the merged-PF backfill. Caches: NPY artifacts under `NPY_SAVE_PATH`.

In [ ]:
ds = load_all_mice(cfg)

# Inject every dataset attribute as a top-level notebook name so legacy
# inline blocks (TFC_cond, Test_B, mappings_all_*, engram_id, ...) work unchanged.
globals().update(vars(ds))

print(f"PLOTS_DIR  = {ds.PLOTS_DIR}")
print(f"#mice      = {len(ds.mouse_list)}")
print(f"groups     = { {g: len(ms) for g, ms in ds.mice_per_group.items()} }")

## 4. (Optional) Save loaded dataset to disk for fast restart

Pickling lets you reload in a fresh kernel without re-running the per-mouse loop. Session objects are heavy; expect a multi-GB pickle. Only do this once you trust your `cfg`.

In [ ]:
# import pickle
# pickle_path = os.path.join(ds.TFC_cond_savepath, 'ds_cache.pkl')
# with open(pickle_path, 'wb') as f:
#     pickle.dump(ds, f, protocol=pickle.HIGHEST_PROTOCOL)
# print('saved ->', pickle_path)

## 5. Engram sanity panels

In [ ]:
run_engram_sanity_plots(ds, cfg)

## 6. Resolve continuity-constraint parameters

Required before any 2D Bayesian decoder paradigm. With `cfg.continuity_preset='data-driven'` (the default), this collects velocity stats from every open-field session and derives sigmas from them; saves velocity histograms and a params text file under `PLOTS_DIR/TFC_2D_decoding/`.

In [ ]:
cont_params = resolve_continuity_params(ds, cfg)
vel_stats_TFC = cont_params.pop('vel_stats_TFC')
for k, v in cont_params.items():
    print(f"  {k:30s} = {v}")

# Expose as top-level names for legacy inline blocks:
globals().update(cont_params)

## 7. Epoch PV similarity (within-session and cross-session)

In [ ]:
if cfg.plot_epoch_pv_analysis:
    run_epoch_pv(ds, cfg)
if cfg.plot_cross_session_epoch_pv_analysis:
    run_cross_session_epoch_pv(ds, cfg)

## 8. Population PCA trajectory analysis (all variants)

In [ ]:
pop_pca_all = run_population_pca_all_modes(ds, cfg)
pop_pca_results = pop_pca_all['by_mode']['permouse']
POP_PCA_PLOTS_DIR = os.path.join(
    ds.PLOTS_DIR,
    ('Population_PCA_engram' if cfg.want_engram_pop_pca else 'Population_PCA_noengram')
    + (f"_{cfg.which_engram}_permouse" if cfg.want_engram_pop_pca else ''),
)

## 9. Isomap manifold pipeline

In [ ]:
isomap_results = run_isomap(ds, cfg)

## 10. Build decoder paramsets

Two `BayesianDecoderParamset` instances — `raw_params` (raw-S decoder) and `pf_params` (place-field decoder). Continuity sigmas come from cell 6.

If you want hyperparameter optimization (`cfg.optimize_parameters` / `cfg.optimize_pf_parameters`), do that here before building the paramsets — see legacy `SSTCa2_main.py` L3260–3340 for the optimizer blocks; they mutate the paramset in place via `paramset.update_from_dict(...)`.

In [ ]:
from SSTCa2_pipeline import build_raw_paramset, build_pf_paramset, run_2D_decoder, run_2D_PF_decoder, popcurve_dirname

raw_params = build_raw_paramset(cfg, cont_params)
pf_params  = build_pf_paramset(cfg, cont_params)
print('raw_params =', raw_params.to_dict())
print('pf_params  =', pf_params.to_dict())

## 11. Decoder paradigms (A–F)

Paradigm blocks A–F still live inline in `SSTCa2_main.py` (L3770–5550). To run them here, copy each block into its own cell and replace:

| Old (main.py) | New (notebook) |
|---|---|
| `run_2D_decoder_all_mice(train, test, ...)` | `run_2D_decoder(cfg, raw_params, train, test, ...)` |
| `run_2D_PF_decoder_all_mice(...)` | `run_2D_PF_decoder(cfg, pf_params, ...)` |
| `_popcurve_dirname(name)` | `popcurve_dirname(cfg, name)` |
| bare globals (`use_z_score`, `decoder_type`, `ridge_alpha`, ...) | `cfg.use_z_score`, `cfg.decoder_type`, `cfg.ridge_alpha`, ... |

Required local names already in this notebook's globals: session dicts (`TFC_cond`, `Test_B`, ...), crossreg dicts (`TFC_cond_crossreg`, ...), `mapping_*` constants, `mappings_all_*` lists, `mouse_groups`, `mice_per_group`, `PLOTS_DIR`, `PAPER_DIR`, plus `cont_params` from cell 6.

In [ ]:
# Example skeleton (paradigm A: train TFC_cond -> test recall sessions)
#
# train_sessions = [(TFC_cond, 'TFC_cond')]
# test_targets   = [(Test_B,     'Test_B'),
#                   (Test_B_1wk, 'Test_B_1wk'),
#                   (Test_A,     'Test_A'),
#                   (Test_A_1wk, 'Test_A_1wk')]
# save_dir = os.path.join(PLOTS_DIR, popcurve_dirname(cfg, 'bayes_2D_multitgt_train_TFC_cond_recall'))
# run_2D_decoder(
#     cfg, raw_params,
#     train_sessions, test_targets,
#     mouse_groups=mouse_groups,
#     mice_per_group=mice_per_group,
#     PLOTS_DIR=PLOTS_DIR,
#     train_label='TFC_cond',
#     session_str='recall',
#     encoder_period=cfg.encoder_period,
#     decoder_type=cfg.decoder_type,
#     ridge_alpha=cfg.ridge_alpha,
#     # ... remaining paradigm-specific kwargs from main.py ...
# )

## Reloading code after edits

`%autoreload 2` (cell 1) handles most edits. If you re-bind names with `from X import Y`, run this cell to refresh them too.
Reload **bottom-up** (dependencies first).

In [ ]:
import importlib
import SSTCa2_utilities, SSTCa2_sessions, SSTCa2_analysis
import SSTCa2_engram, SSTCa2_engram_sanity
import SSTCa2_population, SSTCa2_isomap, SSTCa2_epoch_analysis, SSTCa2_spatial
import SSTCa2_decoder
import SSTCa2_config, SSTCa2_loader, SSTCa2_pipeline
for _m in (SSTCa2_utilities, SSTCa2_sessions, SSTCa2_analysis,
           SSTCa2_engram, SSTCa2_engram_sanity,
           SSTCa2_population, SSTCa2_isomap, SSTCa2_epoch_analysis, SSTCa2_spatial,
           SSTCa2_decoder,
           SSTCa2_config, SSTCa2_loader, SSTCa2_pipeline):
    importlib.reload(_m)

# Re-import names that were rebound via `from X import Y`:
from SSTCa2_config import PipelineConfig
from SSTCa2_loader import load_all_mice
from SSTCa2_pipeline import (
    resolve_continuity_params, engram_idx_by_mouse, run_engram_sanity_plots,
    run_population_pca, run_population_pca_all_modes,
    run_isomap, run_epoch_pv, run_cross_session_epoch_pv,
)
print('reload complete')